In [ ]:
# Cell 1: Install arc-agi from competition wheels (offline, works in Phase A and B)
import subprocess, sys
wheels = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--no-index', f'--find-links={wheels}',
    'arc-agi', 'python-dotenv'
])
print('[Cell1] arc-agi installed from competition wheels')

In [ ]:
# Cell 2: Write VericodingAgent to /tmp/my_agent.py
agent_code = """import os, sys, random, hashlib
import numpy as np
from agents.agent import Agent
from arcengine import GameAction


class VericodingAgent(Agent):
    """DenseExplorer strategy with D4 Zobrist state dedup."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._step = 0
        self._visited = set()
        # D4 Zobrist table (64x64x16, precomputed)
        rng = np.random.RandomState(42)
        self._zobrist = rng.randint(0, 2**63-1, size=(64, 64, 16), dtype=np.uint64)
        print('[VericodingAgent] init OK (DenseExplorer + D4 Zobrist)')

    def _grid_hash(self, grid):
        """D4 Zobrist: min over 8 dihedral transforms."""
        h, w = grid.shape[:2]
        best = np.uint64(2**64-1)
        for rot in range(4):
            for flip in (False, True):
                g = np.rot90(grid, rot)
                if flip:
                    g = np.fliplr(g)
                # Pad to 64x64
                gh, gw = g.shape[:2]
                padded = np.zeros((64, 64, 1), dtype=np.uint64)
                padded[:gh, :gw, 0] = g
                hval = np.bitwise_xor.reduce(
                    self._zobrist[:gh, :gw, :] * (padded[:gh, :gw, :] > 0)
                )
                hval = np.bitwise_xor.reduce(hval)
                if hval < best:
                    best = hval
        return int(best)

    def choose_action(self, frames, latest_frame) -> GameAction:
        try:
            return self._strategy(frames, latest_frame)
        except Exception as e:
            print(f'[VericodingAgent] error: {e} -> random')
            return self._random_action(latest_frame)

    def _strategy(self, frames, latest_frame) -> GameAction:
        self._step += 1
        grid = np.array(latest_frame.frame[0], dtype=np.int32)
        available = latest_frame.available_actions
        if not available:
            return GameAction(1)  # ACTION1 default

        # Hash current state -> cycle detection
        h = self._grid_hash(grid)
        self._visited.add(h)

        # Count non-zero cells
        nz = np.count_nonzero(grid)
        h_, w_ = grid.shape[:2]

        # Complex actions (click) vs simple actions
        has_complex = any(a.is_complex() for a in available)

        if has_complex and nz > 0:
            # Click on most common non-zero color
            flat = grid.flatten()
            nonzero = flat[flat > 0]
            tc = int(np.bincount(nonzero).argmax())
            ys, xs = np.where(grid == tc)
            idx = (self._step * 7) % len(ys)
            action = GameAction(6)  # ACTION6 = click
            action.set_data({'x': int(xs[idx]), 'y': int(ys[idx])})
            return action

        if has_complex:
            # Empty grid — click center
            action = GameAction(6)
            action.set_data({'x': w_//2, 'y': h_//2})
            return action

        # Simple actions: try diverse sequence
        simple = [a for a in available if not a.is_complex()]
        idx = self._step % len(simple)
        return GameAction(simple[idx])

    def _random_action(self, latest_frame) -> GameAction:
        avail = getattr(latest_frame, 'available_actions', None) or []
        if not avail:
            return GameAction(1)
        at = random.choice(avail)
        a = GameAction(at)
        if a.is_complex():
            a.set_data({'x': random.randint(0,63), 'y': random.randint(0,63)})
        return a
"""

with open('/tmp/my_agent.py', 'w') as f:
    f.write(agent_code)
print('[Cell2] VericodingAgent written to /tmp/my_agent.py')

In [ ]:
# Cell 3: Phase B — Competition Rerun (gateway + framework)
import os, subprocess, shutil, sys

if os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
    print('[Cell3] Phase B: competition rerun detected')

    # Wait for gateway sidecar
    subprocess.run([
        'curl', '--fail', '--retry', '999', '--retry-all-errors',
        '--retry-delay', '5', '--retry-max-time', '600',
        'http://gateway:8001/api/games'
    ], check=True)
    print('[Cell3] gateway ready')

    # Copy framework from competition data
    comp = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3'
    fw_src = f'{comp}/ARC-AGI-3-Agents'
    fw_dst = '/kaggle/working/ARC-AGI-3-Agents'
    if os.path.exists(fw_src):
        shutil.copytree(fw_src, fw_dst, dirs_exist_ok=True)
    elif os.path.exists(f'{comp}/arc_agi_3_agents'):
        shutil.copytree(f'{comp}/arc_agi_3_agents', fw_dst, dirs_exist_ok=True)
    print(f'[Cell3] framework copied to {fw_dst}')

    # Install agent into framework
    agents_dir = f'{fw_dst}/agents'
    shutil.copy('/tmp/my_agent.py', f'{agents_dir}/my_agent.py')

    # Register VericodingAgent in framework __init__
    init_content = """from agents.my_agent import VericodingAgent

AVAILABLE_AGENTS = {
    "vericoding": VericodingAgent,
}
"""
    with open(f'{agents_dir}/__init__.py', 'w') as f:
        f.write(init_content)

    # Run framework
    sys.path.insert(0, fw_dst)
    result = subprocess.run(
        [sys.executable, 'main.py', '--agent', 'vericoding'],
        cwd=fw_dst, capture_output=False
    )
    print(f'[Cell3] framework exit code: {result.returncode}')
else:
    print('[Cell3] Phase A: skipping (no competition rerun)')

In [ ]:
# Cell 4: Phase A — Write dummy submission.parquet (required to enable Submit button)
import os
import pandas as pd

if not os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
    print('[Cell4] Phase A: writing dummy submission.parquet')
    pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score']
    ).to_parquet('/kaggle/working/submission.parquet', index=False)
    print('[Cell4] submission.parquet written — click Submit to Competition on Kaggle')
else:
    print('[Cell4] Phase B: submission.parquet handled by framework gateway')